In [13]:
from langchain.chat_models import init_chat_model
from open_deep_research.knowledge import (
    dataset_info,
    abbreviation
)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_experimental.utilities import PythonREPL
import textwrap
import os
# os.load_dotenv()

from open_deep_research.prompts import (
    report_planner_query_writer_instructions,
    report_planner_instructions,
    query_writer_instructions,
    section_writer_instructions_v2,
    section_writer_instructions,
    final_section_writer_instructions,
    final_section_writer_instructions_v2,
    section_grader_instructions,
    section_writer_inputs,
    visualization_instructions,
    sql_instructions,
    sql_grader_instructions,
)
from typing import Annotated, List, TypedDict, Literal
from pydantic import BaseModel, Field
import operator

class SQLResponse(BaseModel):
    sql_script: str = Field(None, description="SQL script if cant retrive output empty string ")
    explaination: str = Field(None, description="explain why cant retrive given data schema, else ouput empty")



llm_flash = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
)

llm_flash = llm_flash.with_structured_output(SQLResponse)
llm_4o = ChatOpenAI(
    model_name="gpt-4o",      # or "gpt-4o-mini" for lower cost / latency
    
    temperature=0.0,
    # Optional:
    # max_tokens=2048,
    # timeout=30,              # seconds
    # base_url="https://api.openai.com/v1",
)
import textwrap
from google.cloud import bigquery

def python_execute(code: str) -> str:
    """
    Execute Python code in a fresh PythonREPL and return stdout / errors.
    """
    import traceback
    import sys

    wrapped = "try:\n"
    wrapped += textwrap.indent(code, "    ")
    wrapped += textwrap.dedent(
        """
        except Exception as e:
            print(f"error: {e}")
    """
    )
    # print(wrapped)
    python_repl = PythonREPL()
    return python_repl.run(wrapped)

def query_bigquery(sql: str, project_id: str = "agentic-ai-463517") -> str:
    """
    Run a BigQuery query in a throw-away PythonREPL sandbox and return its stdout.
    Using `repr(sql)` guarantees the SQL is embedded as a *single* clean string.
    """
    code = f"""
from google.cloud import bigquery
client = bigquery.Client(project={project_id!r})
query_job = client.query({sql!r})
if query_job.result().total_rows == 0:
    print("No results found.")
else:
    print(list(query_job.result()))
    df  = query_job.result().to_dataframe()
    df.to_csv('test.csv')
"""
    return python_execute(code)


# Wrap with the Pydantic schema so every call returns a parsed SQLResponse.
llm_4o = llm_4o.with_structured_output(SQLResponse)
def sql_writer(llm, question):
    print("sql writer.....")
    project_id = "agentic-ai-463517"
    dataset_name = "ghn_data"

    system_instructions_query = sql_instructions.format(
                        question=question,
                        dataset_info=dataset_info,
                        project_id=project_id,
                        dataset_name=dataset_name,
                        last_error="there was no error in the last query",
                        previous_query="",
                    )
        # print(system_instructions_query)
    result =  llm.invoke([SystemMessage(content=system_instructions_query),
                                                            HumanMessage(content=question)])
    print("="*100)
    print(system_instructions_query)
    print(result.sql_script)
    print(result.explaination)
    print("="*100)
    if result.sql_script != "":
        output = query_bigquery(result.sql_script)
        print(output)
        print("="*100)
        for i in range(3):
            print(result)
            print(output)
            if "error" in output:
                system_instructions_query = sql_instructions.format(
                                question=question,
                                dataset_info=dataset_info,
                                project_id=project_id,
                                dataset_name=dataset_name,
                                last_error=output,
                                previous_query=result.sql_script,
                            )
                print(system_instructions_query)
                result =  llm.invoke([SystemMessage(content=system_instructions_query),
                                                                        HumanMessage(content=question)])
                output = query_bigquery(result.sql_script)
                print("="*100)
                print(result.sql_script)
                print(result.explaination)
                print(output)
                print("="*100)
            else:
                break
    else:
        print('Cant write sql')
        print(result.explaination)
        






In [14]:
sql_writer(llm_4o, question="give me all distince name of warehouse?")

sql writer.....


Python REPL can execute arbitrary code. Use with caution.



You are an intelligent AI assistant that generates and executes SQL queries in BigQuery format based on user questions.

Given a natural language question, you must:
1. Generate a syntactically correct BigQuery SQL query to answer the question.
2. Return only a relevant subset of columns based on the question. Avoid SELECT * at all costs.
3. Apply mandatory filters when querying specific tables:
   - If querying the `shipping_order` table, always include:
     WHERE ... AND created_date_partition <= "2030-01-01"
   - If querying the `middle_mile_log` table, always include:
     WHERE ... AND action_date <= "2030-01-01"
4. Use only valid column names that exist in the provided schema. Do not invent or assume columns.
5. Ensure column-table correctness — only reference columns that exist in the table being queried.
6. When possible, order the result by a relevant column to surface the most informative or interesting rows.

Your primary objective is to generate safe, valid, and insightfu

/usr/local/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/usr/local/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


[Row((' Bưu Cục TT Lô 20 Đề Thám-Cao Bằng-Cao Bằng\t',), {'warehouse_name': 0}), Row((' Kho giao hàng Ahamove Long An',), {'warehouse_name': 0}), Row((' NutiFood Chương Dương-Thường Tín-Hà Nội',), {'warehouse_name': 0}), Row(('An Giang',), {'warehouse_name': 0}), Row(('B2B Nha Trang-Phạm Thái Vũ',), {'warehouse_name': 0}), Row(('B2B Đắk Lắk-Nguyễn Thị Cẩm Hoàng',), {'warehouse_name': 0}), Row(('B2B Đắk Lắk-Nguyễn Tiến Điệp',), {'warehouse_name': 0}), Row(('BDG-VinMart+-300ThichQuangDuc',), {'warehouse_name': 0}), Row(('BDG-VinMart+-38/12KhuphoDongB',), {'warehouse_name': 0}), Row(('BDG-VinMart+-416NguyenThiMinhKhai',), {'warehouse_name': 0}), Row(('BDG-VinMart+-52/7BinhDuong2',), {'warehouse_name': 0}), Row(('BDG-VinMart+-544ANguyenTrai',), {'warehouse_name': 0}), Row(('BDG-VinMart+-549ADuongDT743A',), {'warehouse_name': 0}), Row(('BDG-VinMart+-62BisCachMangThangTam',), {'warehouse_name': 0}), Row(('BDG-VinMart+-72BisNguyenVanTiet',), {'warehouse_name': 0}), Row(('BDG-VinMart+-86NgoThi

In [4]:
print(1)

1


In [5]:
print(12312)

12312


In [12]:
try:

    from google.cloud import bigquery
    client = bigquery.Client(project='agentic-ai-463517')
    query_job = client.query("SELECT\n    DATE(created_date) AS order_date,\n    dl.region_name,\n    COUNT(so._id) AS total_orders,\n    COUNT(DISTINCT so.deliver_user) AS num_nvpttt,\n    COUNT(so._id) / COUNT(DISTINCT so.deliver_user) AS orders_per_nvpttt\nFROM\n    `agentic-ai-463517.ghn_data.shipping_order` AS so\nJOIN\n    `agentic-ai-463517.ghn_data.dim_location` AS dl ON so.to_district_id = dl.district_id\nWHERE EXTRACT(YEAR FROM so.created_date) = 2023 AND so.created_date_partition <= '2023-10-01'\nGROUP BY\n    1, 2\nORDER BY\n    1, 2")
    if query_job.result().total_rows == 0:
        print("No results found.")
    else:
        # for row in query_job.result():
        #     print(row)
        print(list(query_job.result()))
        df  = query_job.result().to_dataframe()
        df.to_csv('test.csv')
except Exception as e:
    print("An error occurred while executing the query:")
    print(e)

/usr/local/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


[Row((datetime.date(2023, 4, 1), 'Hà Nội', 1000068, 1141, 876.4837861524978), {'order_date': 0, 'region_name': 1, 'total_orders': 2, 'num_nvpttt': 3, 'orders_per_nvpttt': 4}), Row((datetime.date(2023, 4, 1), 'Hồ Chí Minh', 986466, 1228, 803.3110749185668), {'order_date': 0, 'region_name': 1, 'total_orders': 2, 'num_nvpttt': 3, 'orders_per_nvpttt': 4}), Row((datetime.date(2023, 4, 1), 'Khu vực 1', 2760652, 4353, 634.1952676315185), {'order_date': 0, 'region_name': 1, 'total_orders': 2, 'num_nvpttt': 3, 'orders_per_nvpttt': 4}), Row((datetime.date(2023, 4, 1), 'Khu vực 2', 469605, 509, 922.6031434184675), {'order_date': 0, 'region_name': 1, 'total_orders': 2, 'num_nvpttt': 3, 'orders_per_nvpttt': 4}), Row((datetime.date(2023, 4, 1), 'Khu vực 3', 3250074, 3429, 947.819772528434), {'order_date': 0, 'region_name': 1, 'total_orders': 2, 'num_nvpttt': 3, 'orders_per_nvpttt': 4}), Row((datetime.date(2023, 4, 1), 'Đà Nẵng', 77790, 171, 454.9122807017544), {'order_date': 0, 'region_name': 1, '

/usr/local/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
